# 第二轮：渐进训练与记忆消融
第一轮完整序列准确率为 0/100。本轮先用 2–3 个频带音节训练，再逐步扩展到 3–6 个，并保存曲线、详细错误和记忆消融结果。这仍是合成频带任务，不是自然语言语音。

操作顺序：① 选择 Colab GPU 运行时；② 运行仓库获取；③ 挂载 Drive；④ 运行快速检查；⑤ 正式训练和测试；⑥ 查看结果；⑦ 将 `MyDrive/Parrot/trial2_results.zip` 回传。若 Colab 中断，先把已有的 `train_log.csv` 回传。


In [ ]:
from pathlib import Path
import subprocess
repo = Path('/content/Parrot')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/DoctorAwe/Parrot.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
assert (repo / 'trial2.py').is_file() and (repo / 'trial1.py').is_file()
commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('代码提交:', commit)
%cd /content/Parrot


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive/Parrot')
drive_root.mkdir(parents=True, exist_ok=True)
result_dir = drive_root / 'trial2_results'
print('结果目录:', result_dir)


先检查逐帧接口：同一模型逐帧喂入和整段循环调用应给出相同输出；改变未来帧不应影响过去输出。当前帧是合成频带表示，不是从真实麦克风波形提取的帧。随后做两步训练快速检查。


In [ ]:
import torch
from trial2 import StreamMemory
torch.manual_seed(7)
m = StreamMemory().eval()
x = torch.zeros(1, 12, 9); x[:, :, 0] = 1; x[:, 2, 0] = 0; x[:, 2, 3] = 1
with torch.no_grad():
    whole = m(x, feedback_gain=0.15, delay=3)
    state = m.initial_state(1, x.device, x.dtype, delay=3)
    frames = []
    for t in range(x.shape[1]):
        output, state = m.step(x[:, t], state, feedback_gain=0.15)
        frames.append(output)
    online = torch.stack(frames, dim=1)
    changed = x.clone(); changed[:, 9, :] = 0; changed[:, 9, 8] = 1
    alternate = m(changed, feedback_gain=0.15, delay=3)
assert torch.allclose(whole, online, atol=1e-6)
assert torch.allclose(whole[:, :9], alternate[:, :9], atol=1e-6)
print('逐帧调用一致；未来帧未影响过去输出')


In [ ]:
!python trial2.py --smoke --batch 4 --out /content/Parrot/trial2_smoke


正式训练默认 1800 步。Colab 运行时有时会断开，建议在开始前确认 GPU 已启用。结果在训练完成后写入 Drive；训练过程中的日志也会显示在单元格输出中。


In [ ]:
!python trial2.py --steps 1800 --batch 24 --eval 100 --out {result_dir}


In [ ]:
import json
from IPython.display import Audio, display
report = json.loads((result_dir / 'metrics.json').read_text())
report['git_commit'] = commit
(result_dir / 'metrics.json').write_text(json.dumps(report, indent=2))
print(json.dumps(report, ensure_ascii=False, indent=2))
for name in ('input.wav', 'target.wav', 'model.wav'):
    print(name)
    display(Audio(filename=str(result_dir / name)))


将包含 `metrics.json`、`train_log.csv`、`examples.json`、模型权重和音频的压缩包回传。评测重点是完整序列准确率、编辑距离、起始时间、错误发声率，以及去掉可塑性连接或清除活动后的性能变化。


In [ ]:
import shutil
archive = shutil.make_archive(str(drive_root / 'trial2_results'), 'zip', root_dir=str(drive_root), base_dir='trial2_results')
print('请回传:', archive)
